# Celda 1 - Imports

In [1]:
import requests
import logging
import sqlite3
import time
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin, quote

URL = "https://books.toscrape.com/catalogue/page-1.html"

logging.basicConfig(
    filename="scrape.log",                
    filemode="w",                         
    level=logging.INFO,                   
    format="%(asctime)s - %(levelname)s - %(message)s"  
)

## Celda 2 — Funciones auxiliares

In [2]:
def clean_title(title):
    """Remove subtitles and series info for better API matching."""
    for char in [':', '(']:
        if char in title:
            title = title[:title.index(char)]
    return title.strip()

def extract_year(raw):
    """Extract 4-digit year from any date string format."""
    if raw is None:
        return None
    for word in str(raw).split():
        if len(word) >= 4:
            digits = ''.join(c for c in word if c.isdigit())
            if len(digits) == 4:
                return int(digits)
    return None

def extract_country_from_bio(bio):
    """
    Extract country from author bio text.
    Supports both English and Spanish nationality adjectives.
    """
    if bio is None:
        return None
    
    # Map of nationality keywords (English + Spanish) to country names
    country_map = {
        # English
        "american"      : "United States",
        "british"       : "United Kingdom",
        "english"       : "United Kingdom",
        "scottish"      : "Scotland",
        "irish"         : "Ireland",
        "french"        : "France",
        "german"        : "Germany",
        "spanish"       : "Spain",
        "italian"       : "Italy",
        "russian"       : "Russia",
        "japanese"      : "Japan",
        "canadian"      : "Canada",
        "australian"    : "Australia",
        "mexican"       : "Mexico",
        "argentinian"   : "Argentina",
        "argentine"     : "Argentina",
        "colombian"     : "Colombia",
        "chilean"       : "Chile",
        "brazilian"     : "Brazil",
        "portuguese"    : "Portugal",
        "swedish"       : "Sweden",
        "norwegian"     : "Norway",
        "danish"        : "Denmark",
        "dutch"         : "Netherlands",
        "polish"        : "Poland",
        "czech"         : "Czech Republic",
        "indian"        : "India",
        "chinese"       : "China",
        "korean"        : "South Korea",
        # Spanish
        "estadounidense": "United States",
        "americano"     : "United States",
        "americana"     : "United States",
        "británico"     : "United Kingdom",
        "británica"     : "United Kingdom",
        "inglés"        : "United Kingdom",
        "inglesa"       : "United Kingdom",
        "francés"       : "France",
        "francesa"      : "France",
        "alemán"        : "Germany",
        "alemana"       : "Germany",
        "español"       : "Spain",
        "española"      : "Spain",
        "italiano"      : "Italy",
        "italiana"      : "Italy",
        "ruso"          : "Russia",
        "rusa"          : "Russia",
        "japonés"       : "Japan",
        "japonesa"      : "Japan",
        "canadiense"    : "Canada",
        "australiano"   : "Australia",
        "australiana"   : "Australia",
        "mexicano"      : "Mexico",
        "mexicana"      : "Mexico",
        "argentino"     : "Argentina",
        "argentina"     : "Argentina",
        "colombiano"    : "Colombia",
        "colombiana"    : "Colombia",
        "chileno"       : "Chile",
        "chilena"       : "Chile",
        "brasileño"     : "Brazil",
        "brasileña"     : "Brazil",
        "irlandés"      : "Ireland",
        "irlandesa"     : "Ireland",
        "escocés"       : "Scotland",
        "escocesa"      : "Scotland",
        "israeli"       : "Israel",
        "welsh"         : "United Kingdom",
        "south african" : "South Africa",
        "new zealand"   : "New Zealand"
    }
    
    bio_lower = bio.lower()
    for keyword, country in country_map.items():
        if keyword in bio_lower:
            return country
    
    return None

## Celda 3 — Función de API

In [ ]:
# Simple cache to avoid duplicate API calls for the same title
author_cache = {}

def get_author_from_wikipedia(author_name):
    """
    Fallback using Wikipedia REST API.
    Requires User-Agent header — Wikipedia blocks requests without it.
    Extracts country and birth year from the page summary.
    """
    try:
        url  = f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(author_name)}"
        resp = requests.get(
            url,
            timeout=5,
            headers={"User-Agent": "BookScraper/1.0 (educational project)"}  # obligatorio
        )
        
        if resp.status_code != 200:
            return {"country": None, "birth_year": None}

        data = resp.json()
        
        # Combinar extract + description para maximizar info disponible
        extract     = data.get("extract", "") or ""
        description = data.get("description", "") or ""
        text        = (extract + " " + description).lower()

        # Buscar cualquier año entre 1800 y 2024 — más flexible que el patrón anterior
        year_match = re.search(r"\b(1[89]\d{2}|20[0-2]\d)\b", text)
        birth_year = int(year_match.group(1)) if year_match else None

        country = extract_country_from_bio(text)

        return {"country": country, "birth_year": birth_year}

    except Exception as e:
        logging.warning(f"Wikipedia fallback failed for '{author_name}': {e}")
        return {"country": None, "birth_year": None}
    
def get_author(title):
    """
    Query OpenLibrary API to enrich book data with author metadata.
    Uses a local cache to avoid redundant API calls.
    Returns a dict with author fields, all None if not found.
    """
    # Fallback dict returned when author is not found or an error occurs
    empty_result = {
        "author_name"   : None,
        "author_id"     : None,
        "birth_year"    : None,
        "country"       : None,
        "work_count"    : None,
        "source"        : "OpenLibrary API"
    }
    
    cleaned = clean_title(title)
    
    # Return cached result if available
    if cleaned in author_cache:
        logging.info(f"Cache hit for '{cleaned}'")
        return author_cache[cleaned]
    
    url = "https://openlibrary.org/search.json"
    try:
        response = requests.get(url, params={"title": cleaned, "limit": 1}, timeout=5)
        response.raise_for_status()
        datos = response.json()
        
        if not datos.get("docs"):
            logging.warning(f"No author found for '{title}'")
            author_cache[cleaned] = empty_result
            return empty_result
        
        book       = datos["docs"][0]
        author_key = book.get("author_key", [None])[0]
        author_name= book.get("author_name", [None])[0]

        birth_year = None
        country    = None
        work_count = None
        
        # Fetch detailed author info using author_key
        if author_key:
            author_url      = f"https://openlibrary.org/authors/{author_key}.json"
            author_response = requests.get(author_url, timeout=5)
            
            if author_response.status_code == 200:
                author_data = author_response.json()
                
                # Normalize birth year to integer
                birth_year  = extract_year(author_data.get("birth_date", None))
                
                # Try birth_place first, then fall back to parsing the bio text
                birth_place = author_data.get("birth_place", None)
                if birth_place:
                    country = birth_place.split(",")[-1].strip()
                else:
                    bio = author_data.get("bio", None)
                    if isinstance(bio, dict):
                        bio = bio.get("value", None)
                    country = extract_country_from_bio(bio)
                
                # Después de extraer birth_year y country de OpenLibrary
                if author_name and (not country or not birth_year):
                    author_name = author_name.strip()
                    logging.info(f"Trying Wikipedia for '{author_name}'...")
                    wiki_data = get_author_from_wikipedia(author_name)
                    logging.info(f"Wikipedia result: {wiki_data}")
                    
                    if not country and wiki_data["country"]:
                        country = wiki_data["country"]
                    
                    if not birth_year and wiki_data["birth_year"]:
                        birth_year = wiki_data["birth_year"]
                    
                    if wiki_data["country"] or wiki_data["birth_year"]:
                        logging.info(f"Wikipedia used for '{author_name}' → {wiki_data}")
                
                
                # Fetch total work count from separate endpoint
                works_url      = f"https://openlibrary.org/authors/{author_key}/works.json"
                works_response = requests.get(works_url, timeout=5)
                if works_response.status_code == 200:
                    work_count = works_response.json().get("size", None)
        
        logging.info(f"Author found for '{title}': {author_name} | country: {country} | birth_year: {birth_year}")
        
        result = {
            "author_name"   : author_name,
            "author_id"     : author_key,
            "birth_year"    : birth_year,
            "country"       : country,
            "work_count"    : work_count,
            "source"        : "OpenLibrary API"
        }
        
        # Store in cache before returning
        author_cache[cleaned] = result
        return result
        
    except requests.exceptions.HTTPError as e:
        logging.error(f"HTTP error fetching author for '{title}': {e}")
        return empty_result
    except requests.exceptions.RequestException as e:
        logging.error(f"Network error fetching author for '{title}': {e}")
        return empty_result

## Celda 4 — Funciones de scraping

In [ ]:
def fetch_page(url):
    """
    Fetch the HTML content of a given URL.
    Uses headers + higher timeout to reduce connection issues.
    """
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        response = requests.get(url, headers=headers, timeout=10)  # headers ahora se pasa
        response.raise_for_status()
        logging.info(f"Fetched page successfully: {url}")
        return BeautifulSoup(response.text, "html.parser")
    
    except requests.exceptions.HTTPError as e:
        logging.error(f"HTTP error fetching {url}: {e}")
        return None
    
    except requests.exceptions.RequestException as e:
        logging.error(f"General error fetching {url}: {e}")
        return None

def extract_books(soup, current_url):
    """
    Extract book information from a catalogue page.
    For each book, fetches its detail page for the category
    and calls the OpenLibrary API for author enrichment.
    Returns a list of book dicts.
    """
    books_data = []
    books = soup.find_all("article", class_="product_pod")
    
    for book in books:
        # Basic info from the catalogue listing
        title  = book.h3.a['title']
        price  = book.find("p", class_="price_color").text
        price  = price.encode("latin-1").decode("utf-8")
        price  = float(price.replace("£", "").strip())
        rating = book.find("p", class_="star-rating")["class"][1]
        
        # Enrich with author metadata from OpenLibrary
        author_info = get_author(title)
        
        # Build absolute URL to the book detail page
        relative_link = book.h3.a["href"]
        link          = urljoin(current_url, relative_link)
        
        # Fetch detail page to get the category from breadcrumb
        book_soup = fetch_page(link)
        if book_soup is None:
            category = "Unknown"
        else:
            category = book_soup.find("ul", class_="breadcrumb").find_all("a")[2].text
        
        # Small delay to avoid hammering the server
        time.sleep(1)
        
        books_data.append({
            "title"      : title,
            "price"      : price,
            "rating"     : rating,
            "category"   : category,
            "author"     : author_info["author_name"],
            "author_id"  : author_info["author_id"],
            "birth_year" : author_info["birth_year"],
            "country"    : author_info["country"],
            "work_count" : author_info["work_count"],
            "source"     : author_info["source"],
            "link"       : link
        })
    
    return books_data

def get_next_page(soup, current_url):
    """
    Check if there is a 'next' button and return its absolute URL.
    Returns None if no next page exists.
    """
    next_button = soup.find("li", class_="next")
    if next_button:
        return urljoin(current_url, next_button.a["href"])
    return None

## Celda 5 — Base de datos

In [5]:
def create_database():
    """
    Create the SQLite database and all required tables.
    Uses IF NOT EXISTS so it's safe to call multiple times.
    """
    conn = sqlite3.connect("books.db")
    cursor = conn.cursor()
    cursor.executescript("""
        CREATE TABLE IF NOT EXISTS categories (
            id   INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE NOT NULL
        );
        CREATE TABLE IF NOT EXISTS authors (
            id               INTEGER PRIMARY KEY AUTOINCREMENT,
            name             TEXT,
            birth_year       INTEGER,
            country          TEXT,
            external_api_id  TEXT UNIQUE,
            total_known_works INTEGER,
            api_source       TEXT,
            created_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
        CREATE TABLE IF NOT EXISTS books (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            title       TEXT UNIQUE NOT NULL,
            price       REAL,
            rating      INTEGER,
            category_id INTEGER,
            FOREIGN KEY (category_id) REFERENCES categories(id)
        );
        CREATE TABLE IF NOT EXISTS book_author (
            book_id   INTEGER,
            author_id INTEGER,
            PRIMARY KEY (book_id, author_id),
            FOREIGN KEY (book_id)   REFERENCES books(id),
            FOREIGN KEY (author_id) REFERENCES authors(id)
        );
    """)
    conn.commit()
    return conn

RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def insert_book(conn, book):
    """
    Insert a single book (and its category/author) into the database.
    Uses INSERT OR IGNORE to safely skip duplicates on re-runs.
    """
    cursor = conn.cursor()
    
    # Insert category if it doesn't exist and retrieve its id
    cursor.execute("INSERT OR IGNORE INTO categories (name) VALUES (?)", (book["category"],))
    cursor.execute("SELECT id FROM categories WHERE name = ?", (book["category"],))
    cat_id = cursor.fetchone()[0]
    
    # Insert book and retrieve its id
    cursor.execute("""
        INSERT OR IGNORE INTO books (title, price, rating, category_id)
        VALUES (?, ?, ?, ?)
    """, (book["title"], book["price"], RATING_MAP.get(book["rating"]), cat_id))
    cursor.execute("SELECT id FROM books WHERE title = ?", (book["title"],))
    book_id = cursor.fetchone()[0]
    
    # Insert author and link to book if author data is available
    if book["author"]:
        cursor.execute("""
            INSERT OR IGNORE INTO authors
            (name, birth_year, country, external_api_id, total_known_works, api_source)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (book["author"], book["birth_year"], book["country"],
              book["author_id"], book["work_count"], book["source"]))
        
        cursor.execute(
            "SELECT id FROM authors WHERE external_api_id = ? OR name = ?",
            (book["author_id"], book["author"])
        )
        author_id = cursor.fetchone()[0]
        
        # Insert many-to-many relationship
        cursor.execute("INSERT OR IGNORE INTO book_author VALUES (?, ?)", (book_id, author_id))
    
    conn.commit()

## Celda 6 — Ejecución

In [ ]:
def scrape_books(url):
    """
    Main pipeline: scrape all pages, enrich with API, persist to DB.
    Saves each page to the DB immediately to avoid data loss on failure.
    """
    conn        = create_database()
    all_books   = []   # acumular todos para el reporte final
    total_books = 0
    page_number = 1
    
    while url:
        print(f"🔍 Scrapeando página {page_number}... ({url})")
        
        soup = fetch_page(url)
        if soup is None:
            print(f"⚠️ No se pudo fetchear página {page_number}, terminando.")
            break
        
        books = extract_books(soup, url)
        
        # Persist each book immediately to avoid data loss on crash
        for book in books:
            insert_book(conn, book)
        
        all_books.extend(books)
        total_books += len(books)
        print(f"✅ Página {page_number} — {len(books)} libros — Total: {total_books} — 💾 guardado en DB")
        
        url = get_next_page(soup, url)
        page_number += 1
    
    # Final enrichment report using all scraped books
    con_autor = sum(1 for b in all_books if b["author"] is not None)
    con_pais  = sum(1 for b in all_books if b["country"] is not None)
    
    print(f"\n🎉 Scraping finalizado — {page_number-1} páginas — {total_books} libros")
    print(f"📊 Reporte de enriquecimiento:")
    print(f"   Con autor   : {con_autor}/{total_books} ({con_autor/total_books*100:.1f}%)")
    print(f"   Con país    : {con_pais}/{total_books}  ({con_pais/total_books*100:.1f}%)")
    print(f"   Sin autor   : {total_books - con_autor} registros con NULL")
    print(f"   Sin país    : {total_books - con_pais} registros con NULL")
    
    conn.close()

scrape_books(URL)

🔍 Scrapeando página 1... (https://books.toscrape.com/catalogue/page-1.html)
✅ Página 1 — 20 libros — Total: 20 — 💾 guardado en DB
🔍 Scrapeando página 2... (https://books.toscrape.com/catalogue/page-2.html)
✅ Página 2 — 20 libros — Total: 40 — 💾 guardado en DB
🔍 Scrapeando página 3... (https://books.toscrape.com/catalogue/page-3.html)
✅ Página 3 — 20 libros — Total: 60 — 💾 guardado en DB
🔍 Scrapeando página 4... (https://books.toscrape.com/catalogue/page-4.html)


KeyboardInterrupt: 